# Ablation Variant A: SmallBERT + Continuous Head (MSE)

**Core Question**: Is the Transformer's fragmented manifold caused by the
attention matrix, or simply because you forced it to output 256 discrete categorical bins?

**The Goal**: Prove that even if you give the Transformer every mathematical advantage
to model a continuous system, the internal attention graph still pays the
Geometric Alignment Tax.

## Design

- Strip off the categorical cross-entropy head from SmallBERT
- Replace it with a **linear projection to a continuous scalar**, trained via **MSE**
- **Input**: discrete 256-bin tokens (simulating biological sequences)
- **Target**: raw continuous ODE floats (NOT requantized staircase)
- Train on the same 3 synthetic datasets (waveform, oscillator, lorenz)
- Run identical Mutation Walk + Shesha Procrustes evaluations

### Critical Design Decision: Dual-Return Generators

The generators return BOTH discrete tokens (model input) AND raw continuous
values (MSE target). This avoids the fatal data-leakage where dividing
discrete tokens by `n_bins-1` creates a quantized staircase target that
trains the continuous head to reproduce discrete fractures.

**Expected Result**: The latent manifold will **still fracture** during the Mutation Walk,
even though the output is truly continuous. This proves the gap is architectural
(attention), not output-head-induced.

## Prerequisites

- Upload `evaluation_harness.py` to `/content/`
- GPU runtime required
- Run after `Architectural_Control_Experiment.ipynb` (for baseline comparison)

---

In [ ]:
# Install Dependencies

print('Installing core dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy

print('\nBuilding mamba-ssm from source...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK (native CUDA)')
except ImportError:
    print('mamba-ssm: FAILED -- will use pure-PyTorch SSM fallback')

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

import os, sys, gc, time
import numpy as np
import pandas as pd
sys.path.insert(0, '.')

OUTPUT_BASE = './results/ablation_variant_a_continuous/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Same params as baseline
SEQ_LEN    = 512
N_BINS     = 256
N_TRAIN    = 50_000
N_EVAL     = 2_000
SEED       = 320
PAD_TOKEN  = N_BINS
MASK_TOKEN = N_BINS + 1
VOCAB_SIZE = N_BINS + 2

D_MODEL    = 256
N_LAYERS   = 4
N_HEADS    = 4
FFN_DIM    = 1024
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

LR         = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS     = 20
BATCH_SIZE = 64
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]
DATASETS = ['waveform', 'oscillator', 'lorenz']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Ablation: Variant A -- Continuous Head (MSE)')
print(f'Output: {OUTPUT_BASE}')

In [ ]:
# DUAL-RETURN Dataset Generators
#
# Each generator returns: (discrete_tokens, continuous_values, global_range)
# - discrete_tokens: int64 (N, SEQ_LEN) -- 0..255 bin indices for model INPUT
# - continuous_values: float32 (N, SEQ_LEN) -- raw ODE floats [0, 1] for MSE TARGET
# - global_range: (min, max) -- dataset-wide range for consistent discretization
#
# Uses dataset-global min/max for discretization and normalization.

from scipy.integrate import solve_ivp


def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    '''Convert continuous values to discrete bin indices.
    Uses global min/max when provided.
    '''
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, n_bins // 2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.clip((normed * (n_bins - 1)).astype(np.int64), 0, n_bins - 1)


def normalize_to_01(values, global_min=None, global_max=None):
    '''Normalize continuous values to [0, 1] without quantization.
    Uses global min/max when provided.
    '''
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, 0.5, dtype=np.float32)
    return np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0).astype(np.float32)


def generate_waveforms(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    '''Returns: (discrete_tokens, continuous_values, global_range)'''
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, seq_len, endpoint=False)
    raw_signals = []
    for _ in range(n_sequences):
        n_sines = rng.integers(3, 6)
        signal = np.zeros(seq_len)
        for _ in range(n_sines):
            signal += rng.uniform(0.1, 1.0) * np.sin(
                2 * np.pi * rng.uniform(0.5, 50.0) * t + rng.uniform(0, 2*np.pi))
        raw_signals.append(signal)

    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    gmin, gmax = global_range
    tokens_list = [discretize(s, global_min=gmin, global_max=gmax) for s in raw_signals]
    contin_list = [normalize_to_01(s, global_min=gmin, global_max=gmax) for s in raw_signals]
    return (np.array(tokens_list, dtype=np.int64),
            np.array(contin_list, dtype=np.float32),
            global_range)


def generate_oscillators(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    '''Returns: (discrete_tokens, continuous_values, global_range)'''
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 4, seq_len, endpoint=False)
    raw_signals = []
    for _ in range(n_sequences):
        A = rng.uniform(0.5, 2.0)
        gamma = rng.uniform(0.2, 2.0)
        omega = rng.uniform(2.0, 20.0)
        phi = rng.uniform(0, 2 * np.pi)
        signal = A * np.exp(-gamma * t) * np.cos(omega * t + phi)
        raw_signals.append(signal)

    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    gmin, gmax = global_range
    tokens_list = [discretize(s, global_min=gmin, global_max=gmax) for s in raw_signals]
    contin_list = [normalize_to_01(s, global_min=gmin, global_max=gmax) for s in raw_signals]
    return (np.array(tokens_list, dtype=np.int64),
            np.array(contin_list, dtype=np.float32),
            global_range)


def _lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]


def generate_lorenz(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    '''Returns: (discrete_tokens, continuous_values, global_range)'''
    rng = np.random.default_rng(seed)
    t_span = (0, 25)
    t_eval = np.linspace(*t_span, seq_len)
    raw_signals = []
    for _ in range(n_sequences):
        x0 = rng.uniform(-15, 15)
        y0 = rng.uniform(-15, 15)
        z0 = rng.uniform(10, 40)
        sol = solve_ivp(_lorenz_rhs, t_span, [x0, y0, z0],
                        t_eval=t_eval, method='RK45', max_step=0.05)
        if sol.success and len(sol.y[0]) == seq_len:
            raw = sol.y[0]
        else:
            x0 += rng.uniform(-1, 1)
            sol2 = solve_ivp(_lorenz_rhs, t_span, [x0, y0, z0],
                             t_eval=t_eval, method='RK45', max_step=0.01)
            raw = sol2.y[0][:seq_len]
            if len(raw) < seq_len:
                raw = np.pad(raw, (0, seq_len - len(raw)), mode='edge')
        raw_signals.append(raw)

    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    gmin, gmax = global_range
    tokens_list = [discretize(s, global_min=gmin, global_max=gmax) for s in raw_signals]
    contin_list = [normalize_to_01(s, global_min=gmin, global_max=gmax) for s in raw_signals]
    return (np.array(tokens_list, dtype=np.int64),
            np.array(contin_list, dtype=np.float32),
            global_range)


GENERATORS = {
    'waveform': generate_waveforms,
    'oscillator': generate_oscillators,
    'lorenz': generate_lorenz,
}

# Storage for global ranges
GLOBAL_RANGES = {}

# Verify dual-return
for name, gen_fn in GENERATORS.items():
    tokens, contin, grange = gen_fn(5, seed=1991)
    assert tokens.shape == (5, SEQ_LEN), f'{name}: tokens shape {tokens.shape}'
    assert contin.shape == (5, SEQ_LEN), f'{name}: contin shape {contin.shape}'
    assert tokens.dtype == np.int64
    assert contin.dtype == np.float32
    recon = tokens.astype(np.float32) / (N_BINS - 1)
    max_diff = np.abs(contin - recon).max()
    print(f'{name:12s}: tokens={tokens.shape}  contin={contin.shape}  '
          f'max_quant_err={max_diff:.6f}  range=({grange[0]:.3f}, {grange[1]:.3f})')

print('\nDual-return generators ready (global discretization + raw continuous targets)')


In [ ]:
# Perturbation Suite (operates on discrete tokens only)

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: np.ndarray
    params: dict = field(default_factory=dict)
    description: str = ''

def noise_perturb(sequences, rate, rng, n_bins=N_BINS):
    out = sequences.copy()
    noise_std = n_bins * 0.10
    mask = rng.random(out.shape) < rate
    noise = rng.normal(0, noise_std, size=out.shape).astype(np.int64)
    out[mask] = np.clip(out[mask] + noise[mask], 0, n_bins - 1)
    return out

def time_reverse(sequences):
    return sequences[:, ::-1].copy()

class ContinuousPerturbationSuite:
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES

    def run_all(self, sequences):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate * 100)}pct'
            results[name] = PerturbedSet(
                name=name, category='noise',
                sequences=noise_perturb(sequences, rate, self.rng),
                params={'noise_rate': rate},
                description=f'Value noise at {rate*100:.0f}%',
            )
        results['time_reversal'] = PerturbedSet(
            name='time_reversal', category='reversal',
            sequences=time_reverse(sequences),
            params={}, description='Time reversal',
        )
        return results

print('ContinuousPerturbationSuite ready')

In [ ]:
# Model Definitions
#
# SmallBERT_Continuous: Same architecture as SmallBERT but with:
#   - Linear head -> scalar output (instead of vocab_size logits)
#   - Trained with MSE loss on RAW continuous ODE values
#   - Input still uses token embeddings (discretized bins)
#
# SmallMamba_Continuous: Same modification for fair comparison

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class SmallBERT_Continuous(nn.Module):
    """SmallBERT with continuous scalar output head (MSE loss).
    
    Architecture identical to baseline SmallBERT except:
    - head: Linear(d_model, 1) instead of Linear(d_model, vocab_size)
    - Output is a continuous scalar per position
    """

    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN,
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        # KEY CHANGE: continuous scalar head
        self.head    = nn.Linear(d_model, 1)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, x, return_hidden=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        causal_mask = self._causal_mask(L, x.device)
        h = self.encoder(h, mask=causal_mask)
        h = self.norm(h)
        out = self.head(h).squeeze(-1)  # (B, L)
        if return_hidden:
            return out, h
        return out


# SmallMamba_Continuous uses identical structure
if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock

    class SmallMamba_Continuous(nn.Module):
        """SmallMamba with continuous scalar output head."""
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)
            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(MambaBlock(d_model=d_model, d_state=d_state,
                                             d_conv=d_conv, expand=expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, 1)  # continuous scalar
            self._init_weights()

        def _init_weights(self):
            for name, p in self.named_parameters():
                if 'tok_emb' in name or 'pos_emb' in name:
                    nn.init.normal_(p, std=0.02)
                elif p.dim() > 1 and 'head' in name:
                    nn.init.xavier_uniform_(p)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            out = self.head(h).squeeze(-1)
            if return_hidden:
                return out, h
            return out
else:
    class SimpleSSMLayer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            d_inner = d_model * expand
            self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
            self.conv1d   = nn.Conv1d(d_inner, d_inner, d_conv,
                                      padding=d_conv - 1, groups=d_inner)
            self.x_proj   = nn.Linear(d_inner, d_state * 2, bias=False)
            self.dt_proj  = nn.Linear(d_state, d_inner, bias=True)
            self.A_log    = nn.Parameter(torch.log(torch.randn(d_inner, d_state).abs() + 1e-4))
            self.D        = nn.Parameter(torch.ones(d_inner))
            self.out_proj = nn.Linear(d_inner, d_model, bias=False)

        def forward(self, x):
            B, L, D = x.shape
            xz = self.in_proj(x)
            x_inner, z = xz.chunk(2, dim=-1)
            x_conv = self.conv1d(x_inner.transpose(1, 2))[:, :, :L].transpose(1, 2)
            x_conv = torch.silu(x_conv)
            y = x_conv * torch.silu(z)
            y = y * self.D.unsqueeze(0).unsqueeze(0)
            return self.out_proj(y)

    class SmallMamba_Continuous(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)
            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(SimpleSSMLayer(d_model, d_state, d_conv, expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, 1)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            out = self.head(h).squeeze(-1)
            if return_hidden:
                return out, h
            return out

# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        # No weight tying (vocab_size=258 != d_model=256)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


class SmallStripedHyena_Continuous(nn.Module):
    '''SmallStripedHyena with continuous scalar output head (MSE loss).

    Uses token embeddings for discrete input (same as BERT/Mamba variants),
    NOT a continuous input projection. The whole point of the ablation is
    "same discrete input, different output head."
    '''
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        # Use token embeddings (same as BERT/Mamba continuous variants)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order,
                              is_attention=(i in self.attention_layers),
                              mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)  # continuous scalar output
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena_Continuous: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.drop(self.tok_emb(x))
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        out = self.head(h).squeeze(-1)
        return (out, h) if return_hidden else out


MODEL_CLASSES = {
    'SmallBERT_Continuous': SmallBERT_Continuous,
    'SmallMamba_Continuous': SmallMamba_Continuous,
    'SmallStripedHyena_Continuous': SmallStripedHyena_Continuous,
}
ARCHITECTURES = list(MODEL_CLASSES.keys())

for name, cls in MODEL_CLASSES.items():
    m = cls()
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name:30s}: {n_params/1e6:.2f}M params')
    del m
print('\nModels ready (continuous MSE head)')


In [ ]:
# Training Loop -- Continuous Regression (MSE)
#
# CRITICAL: Uses raw continuous ODE floats as targets, NOT requantized tokens.
#
# Input:  discrete tokens x_0, x_1, ..., x_{L-2}
# Target: raw continuous float c_1, c_2, ..., c_{L-1} (next-step prediction)
#
# The model reads discrete token embeddings but predicts the TRUE continuous
# value at the next position. This is the only valid way to test whether a
# continuous output head changes the manifold geometry.

from torch.utils.data import DataLoader, TensorDataset


def train_model_continuous(model, train_tokens, train_contin, val_tokens, val_contin,
                           arch_name, dataset_name, epochs=EPOCHS,
                           batch_size=BATCH_SIZE, lr=LR,
                           weight_decay=WEIGHT_DECAY, seed=SEED):
    """Train with MSE loss on raw continuous targets.
    
    Args:
        train_tokens: (N, L) int64 -- discrete bin indices (model input)
        train_contin: (N, L) float32 -- raw continuous ODE values [0,1] (target)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = model.to(DEVICE)
    
    # Pack both tokens and continuous values into same dataset
    train_ds = TensorDataset(
        torch.from_numpy(train_tokens).long(),
        torch.from_numpy(train_contin).float(),
    )
    val_ds = TensorDataset(
        torch.from_numpy(val_tokens).long(),
        torch.from_numpy(val_contin).float(),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader))
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    train_losses, val_losses = [], []

    ckpt_path = f'{CKPT_DIR}/{arch_name}_{dataset_name}_seed{seed}_best.pt'
    if os.path.exists(ckpt_path):
        print(f'  Loading existing checkpoint: {ckpt_path}')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        return model, [], []

    print(f'  Training {arch_name} on {dataset_name} (MSE on raw continuous, seed={seed}, {epochs} epochs)...')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for batch_tokens, batch_contin in train_loader:
            batch_tokens = batch_tokens.to(DEVICE)   # (B, L) int64
            batch_contin = batch_contin.to(DEVICE)    # (B, L) float32
            
            # CLM-style: input = tokens[:-1], target = continuous[1:]
            inputs  = batch_tokens[:, :-1]    # discrete token input
            targets = batch_contin[:, 1:]     # RAW continuous target (NOT requantized!)
            
            preds = model(inputs)  # (B, L-1) continuous predictions
            loss = criterion(preds, targets)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            n_batches += 1

        avg_train = epoch_loss / n_batches
        train_losses.append(avg_train)

        model.eval()
        val_loss, val_batches = 0.0, 0
        with torch.no_grad():
            for batch_tokens, batch_contin in val_loader:
                batch_tokens = batch_tokens.to(DEVICE)
                batch_contin = batch_contin.to(DEVICE)
                inputs  = batch_tokens[:, :-1]
                targets = batch_contin[:, 1:]
                preds = model(inputs)
                loss = criterion(preds, targets)
                val_loss += loss.item()
                val_batches += 1

        avg_val = val_loss / max(val_batches, 1)
        val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            elapsed = time.time() - start
            print(f'    Epoch {epoch+1:3d}/{epochs}  '
                  f'train_mse={avg_train:.6f}  val_mse={avg_val:.6f}  '
                  f'best={best_val_loss:.6f}  [{elapsed:.0f}s]')

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)
    elapsed = time.time() - start
    print(f'  Done in {elapsed/60:.1f} min  (best MSE: {best_val_loss:.6f})')
    return model, train_losses, val_losses


print('Training loop ready (MSE on raw continuous ODE targets -- no staircase leakage)')

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness, ModelReport

harness = StabilityHarness(
    window_size=0, metric='cosine', n_splits=30,
    seed=320, max_samples=2500, n_bootstrap=5,
)
print('Shesha StabilityHarness configured (bootstrap=5)')

In [ ]:
# Run All 6 Conditions
#
# Note: perturbations operate on DISCRETE TOKENS (same as baseline).
# The continuous targets are only used during training.
# Embedding extraction uses the trained model on discrete token sequences.

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def extract_embeddings(model, sequences, batch_size=128):
    """Extract mean-pooled hidden states (identical to baseline)."""
    model.eval()
    all_embs = []
    for i in range(0, len(sequences), batch_size):
        batch = torch.from_numpy(sequences[i:i+batch_size]).long().to(DEVICE)
        _, hidden = model(batch, return_hidden=True)
        emb = hidden.mean(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.concatenate(all_embs, axis=0)


print('=' * 70)
print('ABLATION VARIANT A: CONTINUOUS HEAD (MSE on raw ODE floats)')
print('=' * 70)

all_reports = {}
all_detailed_rows = []
training_curves = {}

for ds_name in DATASETS:
    gen_fn = GENERATORS[ds_name]
    print(f"\n{'='*70}")
    print(f'DATASET: {ds_name.upper()}')
    print('=' * 70)

    # Dual-return: tokens + continuous + global range
    train_tokens, train_contin, grange = gen_fn(N_TRAIN, seed=SEED)
    GLOBAL_RANGES[ds_name] = grange
    eval_tokens, eval_contin, _ = gen_fn(N_EVAL, seed=SEED + 1000, global_range=grange)
    print(f'  Global range: ({grange[0]:.3f}, {grange[1]:.3f})')
    
    # Perturbations operate on discrete tokens only
    pert_suite = ContinuousPerturbationSuite(seed=SEED)
    perturbed_sets = pert_suite.run_all(eval_tokens)

    for arch_name in ARCHITECTURES:
        print(f"\n  {'---'*20}")
        print(f'  ARCHITECTURE: {arch_name}')
        print(f'  {"---"*20}')

        condition_key = f'{arch_name}_{ds_name}'
        model = MODEL_CLASSES[arch_name]()
        n_params = sum(p.numel() for p in model.parameters()) / 1e6

        # Train with raw continuous targets
        model, t_losses, v_losses = train_model_continuous(
            model, train_tokens, train_contin,
            eval_tokens, eval_contin,
            arch_name, ds_name, seed=SEED
        )
        training_curves[(ds_name, arch_name)] = (t_losses, v_losses)

        # Extract embeddings from discrete tokens
        cache_clean = f'{CACHE_DIR}/{condition_key}_clean.npy'
        if os.path.exists(cache_clean):
            embeddings_clean = np.load(cache_clean)
        else:
            embeddings_clean = extract_embeddings(model, eval_tokens)
            np.save(cache_clean, embeddings_clean)
        print(f'    Clean embeddings: {embeddings_clean.shape}')

        perturbed_embeddings = {}
        for pert_name, pset in perturbed_sets.items():
            cache_pert = f'{CACHE_DIR}/{condition_key}_{pert_name}.npy'
            if os.path.exists(cache_pert):
                perturbed_embeddings[pert_name] = np.load(cache_pert)
            else:
                perturbed_embeddings[pert_name] = extract_embeddings(model, pset.sequences)
                np.save(cache_pert, perturbed_embeddings[pert_name])

        # Shesha evaluation
        report = harness.evaluate_all_perturbations(
            model_name=condition_key,
            embeddings_clean=embeddings_clean,
            perturbed_dict=perturbed_embeddings,
        )
        all_reports[(ds_name, arch_name)] = report

        for pert_name, metrics in report.perturbation_breakdown().items():
            all_detailed_rows.append({
                'dataset': ds_name, 'architecture': arch_name,
                'variant': 'VariantA_Continuous',
                'n_params_M': round(n_params, 2),
                'perturbation': pert_name,
                'rdm_similarity': metrics.get('rdm_similarity_score', 0),
                'rdm_drift': metrics.get('rdm_drift', 0),
                'pert_stability': metrics.get('perturbation_stability_score', 0),
                'pert_magnitude': metrics.get('perturbation_magnitude', 0),
                'composite': metrics.get('composite_stability', 0),
            })

        summary = report.summary()
        print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
        print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')

        del model
        cleanup_gpu()

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/variant_a_continuous_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nSaved: {detailed_path}')
print(df_detailed.to_string(index=False))